<a href="https://colab.research.google.com/github/514em/assignment2/blob/main/assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.listdir('/content/drive/My Drive/Colab Notebooks/assignment2')

['images',
 'assignment1.ipynb',
 'picture.jpg',
 '3D7BE144-0A71-4F0A-A4CD-C5EA6A22F0C7.jpg']

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks/assignment2/'

# Section 1: Image Stitching [20 points]

In this section, we implement keypoint detection, feature matching, homography estimation, and image stitching using both pyramid blending and linear blending with feathering.

## Q1.1 - Load Images and Convert to Grayscale [3 points]

In [ ]:
# Q1.1: Load all 3 images, convert to grayscale, and display

# Load images from the Q1 folder
view1 = cv2.imread(path + 'images/Q1/view1.jpeg')
view2 = cv2.imread(path + 'images/Q1/view2.jpeg')
view3 = cv2.imread(path + 'images/Q1/view3.jpeg')

# Convert BGR (OpenCV default) to RGB for display
view1_rgb = cv2.cvtColor(view1, cv2.COLOR_BGR2RGB)
view2_rgb = cv2.cvtColor(view2, cv2.COLOR_BGR2RGB)
view3_rgb = cv2.cvtColor(view3, cv2.COLOR_BGR2RGB)

# Convert to grayscale
view1_gray = cv2.cvtColor(view1, cv2.COLOR_BGR2GRAY)
view2_gray = cv2.cvtColor(view2, cv2.COLOR_BGR2GRAY)
view3_gray = cv2.cvtColor(view3, cv2.COLOR_BGR2GRAY)

# Display color and grayscale versions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0, 0].imshow(view1_rgb); axes[0, 0].set_title('View 1 (Color)'); axes[0, 0].axis('off')
axes[0, 1].imshow(view2_rgb); axes[0, 1].set_title('View 2 (Color)'); axes[0, 1].axis('off')
axes[0, 2].imshow(view3_rgb); axes[0, 2].set_title('View 3 (Color)'); axes[0, 2].axis('off')
axes[1, 0].imshow(view1_gray, cmap='gray'); axes[1, 0].set_title('View 1 (Grayscale)'); axes[1, 0].axis('off')
axes[1, 1].imshow(view2_gray, cmap='gray'); axes[1, 1].set_title('View 2 (Grayscale)'); axes[1, 1].axis('off')
axes[1, 2].imshow(view3_gray, cmap='gray'); axes[1, 2].set_title('View 3 (Grayscale)'); axes[1, 2].axis('off')
plt.suptitle('Q1.1 - Loaded Images', fontsize=14)
plt.tight_layout()
plt.show()
print(f'View 1 shape: {view1.shape}')
print(f'View 2 shape: {view2.shape}')
print(f'View 3 shape: {view3.shape}')

## Q1.2 - SIFT Keypoints and Descriptors [3 points]

In [ ]:
# Q1.2: Use SIFT to compute keypoints and descriptors for view1 and view2

# Create SIFT detector
sift = cv2.SIFT_create()

# Compute keypoints and descriptors simultaneously for both images
kp1, desc1 = sift.detectAndCompute(view1_gray, None)
kp2, desc2 = sift.detectAndCompute(view2_gray, None)

print(f'View 1: {len(kp1)} keypoints, descriptor shape: {desc1.shape}')
print(f'View 2: {len(kp2)} keypoints, descriptor shape: {desc2.shape}')

# (a) Visualize keypoints with orientations and scales using DRAW_RICH_KEYPOINTS
# cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS draws circles showing scale and orientation
view1_kp_img = cv2.drawKeypoints(view1_rgb.copy(), kp1, None,
                                   flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
view2_kp_img = cv2.drawKeypoints(view2_rgb.copy(), kp2, None,
                                   flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
ax1.imshow(view1_kp_img)
ax1.set_title(f'View 1 - SIFT Keypoints ({len(kp1)} keypoints)', fontsize=12)
ax1.axis('off')
ax2.imshow(view2_kp_img)
ax2.set_title(f'View 2 - SIFT Keypoints ({len(kp2)} keypoints)', fontsize=12)
ax2.axis('off')
plt.suptitle('Q1.2a - SIFT Keypoints with Orientations and Scales', fontsize=14)
plt.tight_layout()
plt.show()

# (b) Visualize descriptors: overlapping histograms of 10 descriptors from view1
plt.figure(figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for i in range(10):
    plt.plot(np.arange(128), desc1[i], alpha=0.7, color=colors[i], label=f'Descriptor {i+1}')
plt.xlabel('Dimension Index (0-127)')
plt.ylabel('Descriptor Value')
plt.title('Q1.2b - Overlapping Histograms of 10 SIFT Descriptors (View 1)')
plt.legend(loc='upper right', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## Q1.3 - Feature Matching with BFMatcher [2 points]

In [ ]:
# Q1.3: Match descriptors between view1 and view2 using BFMatcher

# Create BFMatcher with L2 norm (appropriate for SIFT) and cross-check for cleaner matches
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)

# Match descriptors: query=desc1 (view1), train=desc2 (view2)
matches = bf.match(desc1, desc2)

# Sort matches by distance (lower = better)
matches = sorted(matches, key=lambda x: x.distance)

print(f'Total matches found: {len(matches)}')
print(f'Top 10 match distances: {[round(m.distance, 2) for m in matches[:10]]}')

# Draw the top 10 best matches
img_matches = cv2.drawMatches(
    view1_rgb, kp1, view2_rgb, kp2, matches[:10], None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

plt.figure(figsize=(20, 8))
plt.imshow(img_matches)
plt.title('Q1.3 - Top 10 Best Matches between View 1 and View 2 (BFMatcher)')
plt.axis('off')
plt.tight_layout()
plt.show()

## Q1.4 - Homography Estimation with RANSAC [2 points]

In [ ]:
# Q1.4: Estimate homography with RANSAC and apply transformation to align view1 with view2

# Extract matching keypoint coordinates
# queryIdx = index in desc1 (view1), trainIdx = index in desc2 (view2)
src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

# Find homography H mapping view1 -> view2 using RANSAC
# ransacReprojThreshold=5.0 pixels: max reprojection error to classify as inlier
H, mask_ransac = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

print(f'Homography matrix H (view1 -> view2):')
print(H)
print(f'\nInliers: {mask_ransac.sum()} / {len(mask_ransac)} matches')

# Determine canvas size to contain both images after warping
h1, w1 = view1_rgb.shape[:2]
h2, w2 = view2_rgb.shape[:2]

# Transform view1 corners to view2's coordinate system
corners_v1 = np.float32([[0,0],[w1,0],[w1,h1],[0,h1]]).reshape(-1,1,2)
corners_v1_warped = cv2.perspectiveTransform(corners_v1, H)

# All corners (view1 warped + view2 in its own frame)
corners_v2 = np.float32([[0,0],[w2,0],[w2,h2],[0,h2]]).reshape(-1,1,2)
all_corners = np.concatenate([corners_v1_warped, corners_v2], axis=0)

[x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
[x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

# Translation offsets so all coordinates are non-negative in the output canvas
offset_x = max(0, -x_min)
offset_y = max(0, -y_min)
canvas_w = x_max - x_min
canvas_h = y_max - y_min

print(f'\nCanvas size: {canvas_w} x {canvas_h}')
print(f'Offset: ({offset_x}, {offset_y})')

# Adjust H with the translation so view1 warps correctly into the canvas
T = np.array([[1, 0, offset_x], [0, 1, offset_y], [0, 0, 1]], dtype=np.float64)
H_adj = T @ H

# Warp view1 to aligned canvas
view1_warped = cv2.warpPerspective(view1_rgb, H_adj, (canvas_w, canvas_h))

# Display the warped view1
plt.figure(figsize=(14, 7))
plt.imshow(view1_warped)
plt.title('Q1.4 - Transformed View 1 (aligned to View 2 coordinate system)')
plt.axis('off')
plt.tight_layout()
plt.show()

## Q1.5 - Pyramid Blending (Laplacian/DoG) to Stitch View 1 and View 2 [4 points]

In [ ]:
# Q1.5: Stitch the aligned view1 with view2 using Laplacian pyramid blending
# We implement the pyramid from scratch using only cv2.pyrDown() and cv2.pyrUp()

def build_gaussian_pyramid(img, levels):
    """Build a Gaussian pyramid by repeatedly downsampling."""
    pyramid = [img.astype(np.float32)]
    for _ in range(levels - 1):
        pyramid.append(cv2.pyrDown(pyramid[-1]))
    return pyramid

def build_laplacian_pyramid(gauss_pyr):
    """
    Build a Laplacian (Difference-of-Gaussians) pyramid from a Gaussian pyramid.
    Each level L[i] = G[i] - pyrUp(G[i+1]), capturing band-pass detail at that scale.
    The top level is simply the coarsest Gaussian image.
    """
    laplacian = []
    for i in range(len(gauss_pyr) - 1):
        h, w = gauss_pyr[i].shape[:2]
        # Upsample the next level to match current level size exactly
        upsampled = cv2.pyrUp(gauss_pyr[i + 1], dstsize=(w, h))
        laplacian.append(gauss_pyr[i] - upsampled)
    laplacian.append(gauss_pyr[-1])  # top level = coarsest Gaussian
    return laplacian

def pyramid_stitch(imgA, imgB, stitch_col, levels=5):
    """
    Stitch two images using Laplacian pyramid blending.
    - Columns < stitch_col are taken from imgA (left image)
    - Columns >= stitch_col are taken from imgB (right image)
    By blending in the Laplacian domain at each scale, the seam is smoothed
    across all frequency bands, avoiding visible transitions.
    """
    imgA = imgA.astype(np.float32)
    imgB = imgB.astype(np.float32)

    # Step 1: Build Gaussian pyramids for both images
    gpA = build_gaussian_pyramid(imgA, levels)
    gpB = build_gaussian_pyramid(imgB, levels)

    # Step 2: Build Laplacian (DoG) pyramids for both images
    lpA = build_laplacian_pyramid(gpA)
    lpB = build_laplacian_pyramid(gpB)

    # Step 3: At each pyramid level, stitch lpA (left) and lpB (right) at scaled stitch_col
    blended_pyr = []
    for i in range(levels):
        h, w = lpA[i].shape[:2]
        # Scale the stitch column to match current pyramid level
        col = min(stitch_col // (2**i), w)
        level = np.zeros_like(lpA[i])
        level[:, :col] = lpA[i][:, :col]
        level[:, col:] = lpB[i][:, col:]
        blended_pyr.append(level)

    # Step 4: Reconstruct by collapsing the blended pyramid
    result = blended_pyr[-1].copy()
    for i in range(levels - 2, -1, -1):
        h, w = blended_pyr[i].shape[:2]
        result = cv2.pyrUp(result, dstsize=(w, h)) + blended_pyr[i]

    return np.clip(result, 0, 255).astype(np.uint8)


# Place view2 in the same canvas as view1_warped
view2_canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
view2_canvas[offset_y:offset_y+h2, offset_x:offset_x+w2] = view2_rgb

# Find the overlap region between the two images
v1_has_content = view1_warped.any(axis=2)
v2_has_content = view2_canvas.any(axis=2)
overlap_mask = v1_has_content & v2_has_content
overlap_cols = np.where(overlap_mask.any(axis=0))[0]

if len(overlap_cols) > 0:
    stitch_col = int(overlap_cols.mean())  # Use midpoint of overlap as seam
    print(f'Overlap region: columns {overlap_cols[0]} to {overlap_cols[-1]}')
    print(f'Stitch column (midpoint of overlap): {stitch_col}')
else:
    stitch_col = canvas_w // 2
    print('No overlap detected, using canvas midpoint as stitch column')

# Apply Laplacian pyramid blending
num_levels = 5
view12 = pyramid_stitch(view1_warped, view2_canvas, stitch_col, levels=num_levels)

# Display the stitched panorama
plt.figure(figsize=(18, 8))
plt.imshow(view12)
plt.title('Q1.5 - Stitched Panorama (view12) using Laplacian Pyramid Blending')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'view12 shape: {view12.shape}')

## Q1.6 - Linear Blending with Feathering to Stitch View12 and View3 [4 points]

In [ ]:
# Q1.6: Repeat SIFT + matching + homography for view12 and view3,
# then stitch using linear blending with feathering

# Convert view12 to grayscale for SIFT
view12_gray = cv2.cvtColor(view12, cv2.COLOR_RGB2GRAY)
view3_gray_q6 = cv2.cvtColor(view3, cv2.COLOR_BGR2GRAY)  # reuse from above
view3_rgb_q6  = cv2.cvtColor(view3, cv2.COLOR_BGR2RGB)

# Compute SIFT keypoints and descriptors for view12 and view3
kp12, desc12 = sift.detectAndCompute(view12_gray, None)
kp3, desc3  = sift.detectAndCompute(view3_gray_q6, None)

print(f'view12: {len(kp12)} keypoints | view3: {len(kp3)} keypoints')

# Match with BFMatcher
bf2 = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
matches23 = bf2.match(desc12, desc3)
matches23 = sorted(matches23, key=lambda x: x.distance)
print(f'Total matches (view12 vs view3): {len(matches23)}')

# Display top 10 matches
img_matches23 = cv2.drawMatches(
    view12, kp12, view3_rgb_q6, kp3, matches23[:10], None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
plt.figure(figsize=(20, 6))
plt.imshow(img_matches23)
plt.title('Top 10 Matches between view12 and view3')
plt.axis('off')
plt.tight_layout()
plt.show()

# Estimate homography: H23 maps keypoints from view12 -> view3
src_pts23 = np.float32([kp12[m.queryIdx].pt for m in matches23]).reshape(-1, 1, 2)
dst_pts23 = np.float32([kp3[m.trainIdx].pt  for m in matches23]).reshape(-1, 1, 2)
H23, mask23 = cv2.findHomography(src_pts23, dst_pts23, cv2.RANSAC, 5.0)
print(f'\nH23 inliers: {mask23.sum()} / {len(mask23)}')

# We want to warp view3 into view12's coordinate system: use H23_inv
H23_inv = np.linalg.inv(H23)

# Determine canvas size to hold both view12 and warped view3
h12, w12 = view12.shape[:2]
h3,  w3  = view3_rgb_q6.shape[:2]

corners_v3 = np.float32([[0,0],[w3,0],[w3,h3],[0,h3]]).reshape(-1,1,2)
corners_v3_in_v12 = cv2.perspectiveTransform(corners_v3, H23_inv)
corners_v12_frame = np.float32([[0,0],[w12,0],[w12,h12],[0,h12]]).reshape(-1,1,2)
all_corners23 = np.concatenate([corners_v3_in_v12, corners_v12_frame], axis=0)

[xmin23, ymin23] = np.int32(all_corners23.min(axis=0).ravel() - 0.5)
[xmax23, ymax23] = np.int32(all_corners23.max(axis=0).ravel() + 0.5)

off_x23 = max(0, -xmin23)
off_y23 = max(0, -ymin23)
cw23 = xmax23 - xmin23
ch23 = ymax23 - ymin23
print(f'Canvas for view123: {cw23} x {ch23}, offset: ({off_x23}, {off_y23})')

# Adjust H23_inv with translation offset
T23 = np.array([[1,0,off_x23],[0,1,off_y23],[0,0,1]], dtype=np.float64)
H23_inv_adj = T23 @ H23_inv

# Warp view3 into view12's coordinate frame
view3_warped = cv2.warpPerspective(view3_rgb_q6, H23_inv_adj, (cw23, ch23))

# Place view12 in the expanded canvas
view12_canvas = np.zeros((ch23, cw23, 3), dtype=np.uint8)
y12e = min(off_y23 + h12, ch23)
x12e = min(off_x23 + w12, cw23)
view12_canvas[off_y23:y12e, off_x23:x12e] = view12[:y12e-off_y23, :x12e-off_x23]

In [ ]:
# Linear blending with feathering for view12 + view3

def linear_blend_feathering(imgA, imgB):
    """
    Blend two same-sized canvas images using linear feathering in the overlap region.
    imgA: left/reference image canvas
    imgB: right/warped image canvas
    - Before overlap: use imgA exclusively
    - In overlap: linearly fade from imgA (alpha=1) to imgB (alpha=0)
    - After overlap: use imgB exclusively
    """
    h, w = imgA.shape[:2]

    # Identify where each image has non-zero content
    a_content = imgA.any(axis=2)
    b_content = imgB.any(axis=2)

    # Find column range of the overlap region
    overlap = a_content & b_content
    overlap_cols = np.where(overlap.any(axis=0))[0]

    if len(overlap_cols) == 0:
        # No overlap: simple composite
        result = imgA.astype(np.float32).copy()
        only_b = (~a_content) & b_content
        result[only_b] = imgB[only_b].astype(np.float32)
        return np.clip(result, 0, 255).astype(np.uint8)

    x_start = int(overlap_cols[0])
    x_end   = int(overlap_cols[-1])
    print(f'Overlap region: columns {x_start} to {x_end} ({x_end - x_start} px wide)')

    # Build a 1D alpha ramp: 1 at x_start -> 0 at x_end (imgA fades out)
    overlap_width = x_end - x_start + 1
    ramp = np.linspace(1.0, 0.0, overlap_width)  # shape: (overlap_width,)

    # Expand to (h, w, 1) for broadcasting
    alpha = np.ones((h, w), dtype=np.float32)
    alpha[:, x_start:x_end+1] = ramp   # transition zone
    alpha[:, x_end+1:] = 0.0            # right of overlap: full imgB
    alpha = alpha[:, :, np.newaxis]      # (h, w, 1)

    # Blend: result = alpha * imgA + (1 - alpha) * imgB
    imgA_f = imgA.astype(np.float32)
    imgB_f = imgB.astype(np.float32)
    result = alpha * imgA_f + (1.0 - alpha) * imgB_f

    return np.clip(result, 0, 255).astype(np.uint8)


# Apply linear blending with feathering
view123 = linear_blend_feathering(view12_canvas, view3_warped)

# Display the final panorama
plt.figure(figsize=(20, 8))
plt.imshow(view123)
plt.title('Q1.6 - Final Panorama (view123)\nView1+View2: Pyramid Blending | View12+View3: Linear Feathering')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Final panorama shape: {view123.shape}')

## Q1.7 - Analysis [2 points]

### Effectiveness of SIFT Keypoints

SIFT (Scale-Invariant Feature Transform) keypoints were highly effective at finding correspondences between the overlapping image pairs. SIFT is designed to be invariant to scale changes, rotation, and moderate illumination differences, making it well-suited for panorama stitching where the same 3D scene is captured from slightly different viewpoints. Each keypoint descriptor is a 128-dimensional histogram of gradient orientations, which provides a rich and distinctive signature.

In the top 10 matches displayed, most matches were accurate (corresponding to the same real-world points). However, a small number of mismatches (outliers) can appear, particularly in texturally repetitive regions (e.g., sky or uniform surfaces) where multiple keypoints have similar descriptors. The use of RANSAC in the homography estimation step helps filter out these outliers and compute a robust transformation from the inlier matches only.

### Pyramid Blending vs. Linear Blending

**Pyramid blending** (Laplacian pyramid / DoG-based) decomposes each image into multiple frequency bands using a Laplacian pyramid, performs the blending at each scale separately, and then reconstructs the result. The key advantage is that the blending width is adapted to each frequency: low-frequency (coarse) components are blended over a wide transition zone, while high-frequency (fine) details are blended over a narrow zone. This effectively eliminates both visible seams (caused by intensity/colour differences) and ghosting artefacts (caused by misalignment at fine scales). The result is a visually seamless blend.

**Linear blending with feathering** applies a simple linear gradient mask in the overlap region: the contribution of each image transitions smoothly from 100% to 0% across the overlap. This eliminates hard seams but can produce ghost artefacts near the seam when there is any misalignment between the two images, because high-frequency content from both images is simultaneously present in the blend zone. It is computationally simpler than pyramid blending and performs well when the images are well-aligned and the overlap is not too wide.

**Conclusion:** Pyramid blending generally produces better results because it handles both low-frequency colour/brightness differences and high-frequency texture/edge mismatches appropriately at each scale. Linear blending is a good approximation when images are very well-registered and the overlap region is narrow, but it tends to show ghosting when there is any residual parallax or misalignment.

# Section 2: Pug Detector [30 points]

In this section we implement a PCA-based Pug detector using eigenfaces. PCA is implemented from scratch using the snapshot method. All further processing uses **grayscale** images only.

## Q2.1 - Load All Q2 Images, Convert to Grayscale, and Display [2 points]

In [ ]:
# Q2.1: Load all images from the Q2 folder (train and val, pugs and loaves)
# (a) Convert to grayscale  (b) Display all images

import glob

# ── Paths (images live under path + 'images/Q2/') ───────────────────────
train_pug_paths  = sorted(glob.glob(path + 'images/Q2/train/pug*.jpg'))
train_loaf_paths = sorted(glob.glob(path + 'images/Q2/train/loaf*.jpg'))
val_pug_paths    = sorted(glob.glob(path + 'images/Q2/val/pug*.jpg'))
val_loaf_paths   = sorted(glob.glob(path + 'images/Q2/val/loaf*.jpg'))

print(f'Training pugs  : {len(train_pug_paths)}')
print(f'Training loaves: {len(train_loaf_paths)}')
print(f'Val pugs       : {len(val_pug_paths)}')
print(f'Val loaves     : {len(val_loaf_paths)}')

# ── Load colour + grayscale ──────────────────────────────────────────────
def load_images(paths):
    """Return (rgb_list, gray_list, name_list) for a list of paths."""
    rgbs, grays, names = [], [], []
    for p in paths:
        bgr  = cv2.imread(p)
        rgbs.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        grays.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY))
        names.append(p.split('/')[-1])
    return rgbs, grays, names

train_pug_rgb,  train_pug_gray,  train_pug_names  = load_images(train_pug_paths)
train_loaf_rgb, train_loaf_gray, train_loaf_names = load_images(train_loaf_paths)
val_pug_rgb,    val_pug_gray,    val_pug_names    = load_images(val_pug_paths)
val_loaf_rgb,   val_loaf_gray,   val_loaf_names   = load_images(val_loaf_paths)

# ── Display helper (Tutorial 7 subplot style) ────────────────────────────
def display_set(rgb_list, gray_list, names, title):
    n = len(rgb_list)
    if n == 0:
        print(f'No images: {title}'); return
    cols = min(n, 5)
    rows_per_type = -(-n // cols)  # ceil division

    # Colour row(s)
    fig, axes = plt.subplots(rows_per_type, cols,
                             figsize=(3*cols, 3*rows_per_type))
    axes = np.array(axes).reshape(rows_per_type, cols)
    for i in range(n):
        r, c = divmod(i, cols)
        axes[r, c].imshow(rgb_list[i])
        axes[r, c].set_title(names[i], fontsize=7)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    for i in range(n, rows_per_type*cols):
        r, c = divmod(i, cols); axes[r, c].axis('off')
    fig.suptitle(title + ' — Colour', fontsize=12)
    plt.tight_layout(); plt.show()

    # Grayscale row(s)
    fig, axes = plt.subplots(rows_per_type, cols,
                             figsize=(3*cols, 3*rows_per_type))
    axes = np.array(axes).reshape(rows_per_type, cols)
    for i in range(n):
        r, c = divmod(i, cols)
        axes[r, c].imshow(gray_list[i], cmap='grey')
        axes[r, c].set_title(names[i], fontsize=7)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    for i in range(n, rows_per_type*cols):
        r, c = divmod(i, cols); axes[r, c].axis('off')
    fig.suptitle(title + ' — Grayscale', fontsize=12)
    plt.tight_layout(); plt.show()

display_set(train_pug_rgb,  train_pug_gray,  train_pug_names,  'Training Pugs')
display_set(train_loaf_rgb, train_loaf_gray, train_loaf_names, 'Training Loaves')
display_set(val_pug_rgb,    val_pug_gray,    val_pug_names,    'Validation Pugs')
display_set(val_loaf_rgb,   val_loaf_gray,   val_loaf_names,   'Validation Loaves')

# ── Resize all grayscale images to a common size (ref = first training pug) ──
IMG_H, IMG_W = train_pug_gray[0].shape[:2]
print(f'\nReference size: {IMG_W} x {IMG_H} px')

def resize_gray(gray_list, h, w):
    return [cv2.resize(g, (w, h), interpolation=cv2.INTER_AREA) for g in gray_list]

train_pug_gray_r  = resize_gray(train_pug_gray,  IMG_H, IMG_W)
train_loaf_gray_r = resize_gray(train_loaf_gray, IMG_H, IMG_W)
val_pug_gray_r    = resize_gray(val_pug_gray,    IMG_H, IMG_W)
val_loaf_gray_r   = resize_gray(val_loaf_gray,   IMG_H, IMG_W)

print(f'Resized — train pugs: {len(train_pug_gray_r)}, '
      f'train loaves: {len(train_loaf_gray_r)}, '
      f'val pugs: {len(val_pug_gray_r)}, '
      f'val loaves: {len(val_loaf_gray_r)}')

## Q2.2 - Mean Pug Face [5 points]

In [ ]:
# Q2.2: Compute and display the mean pug face from the 10 training pug images
# Following Tutorial 7 conventions:
#   - data matrix is D × N  (each image is a COLUMN, shape: pixels × n_images)
#   - mean is stored as a (D, 1) column vector for broadcasting

N_train = len(train_pug_gray_r)          # number of training pugs (10)
D       = IMG_H * IMG_W                  # pixels per image

# ── Step 1: Build the pug data matrix (D × N) ───────────────────────────
# Each column = one flattened pug image, dtype float64
pug_dataset = np.array(
    [img.flatten().astype(np.float64) for img in train_pug_gray_r]
).T                                      # shape: (D, N_train)
print(f'pug_dataset shape: {pug_dataset.shape}  (D={D}, N={N_train})')

# ── Step 2: Compute the mean pug face ────────────────────────────────────
# np.mean over the N images (axis=0 on the resized list, or axis=1 on the dataset)
pug_mean      = np.mean(np.array(train_pug_gray_r, dtype=np.float64), axis=0)  # (H, W)
pug_mean_vec  = pug_mean.flatten()[:, np.newaxis]                               # (D, 1)
print(f'pug_mean shape    : {pug_mean.shape}')
print(f'pug_mean_vec shape: {pug_mean_vec.shape}')
print(f'Intensity range   : [{pug_mean.min():.1f}, {pug_mean.max():.1f}]')

# ── Step 3: Centre the dataset around the mean ───────────────────────────
# pug_mean_vec broadcasts across all N columns
pug_dataset_centered = pug_dataset - pug_mean_vec   # shape: (D, N_train)
print(f'Centred dataset shape: {pug_dataset_centered.shape}')

# ── Step 4: Display individual training pugs (Tutorial 7 style) ─────────
n_cols = 5
plt.figure(figsize=(12, 6))
plt.suptitle('Q2.2 — Training Pug Images (Grayscale)', fontsize=13)
for i, img in enumerate(train_pug_gray_r):
    plt.subplot(2, n_cols, i + 1)
    plt.title(train_pug_names[i], fontsize=7)
    plt.xticks([]); plt.yticks([])
    plt.imshow(img, cmap='grey')
plt.tight_layout()
plt.show()

# ── Step 5: Display the mean pug face ────────────────────────────────────
plt.figure(figsize=(4, 4))
plt.title('Mean Pug Face')
plt.xticks([]); plt.yticks([])
plt.imshow(pug_mean, cmap='grey')
plt.colorbar(fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f'Mean pixel intensity : {pug_mean.mean():.2f}')
print(f'Std of mean face     : {pug_mean.std():.2f}')

## Q2.3 - PCA from Scratch (Snapshot Method) [8 points]

Implement PCA from scratch to compute the eigenvectors and eigenvalues of the covariance matrix for the training pug images using the **snapshot method**.

Following Tutorial 7, we implement two equivalent approaches:
- **Method 1 (eigh):** Eigendecompose the small *N×N* matrix `D.T @ D` (snapshot trick avoids the costly *D×D* covariance matrix), then map back to *D*-dimensional space and normalize.
- **Method 2 (SVD):** Apply `np.linalg.svd` to `D`; eigenvalues = σ², eigenvectors = left singular vectors *U*.

Both methods yield the same eigenfaces (up to sign).

In [ ]:
# Q2.3: PCA from scratch — snapshot method
# Reference: Tutorial 7, Cell 10
#
# Snapshot trick: instead of decomposing the huge (D×D) covariance matrix
# C = D @ D.T, we decompose the tiny (N×N) matrix D.T @ D.
# If v is an eigenvector of D.T @ D with eigenvalue λ,
# then u = D @ v  is an eigenvector of C with the same λ (after normalising).
#
#   Method 1: np.linalg.eigh(D.T @ D)
#     1. Decompose small_cov = D.T @ D  → (eigvals, eigvecs)
#     2. Map to full space: eigvecs_full = D @ eigvecs
#     3. Normalise each column to unit length
#     4. Sort descending by eigenvalue
#
#   Method 2: np.linalg.svd(D)
#     1. D = U Σ V.T  (economy SVD)
#     2. eigenvalues = σ²  (already in descending order)
#     3. eigenfaces  = U   (already in descending order)

D_mat = pug_dataset_centered          # (D_pixels, N_images)
D_pix, N = D_mat.shape
print(f'Centered matrix : {D_mat.shape}  (D_pixels={D_pix}, N_images={N})')

# ══════════════════════════════════════════════════════════════════════
# Method 1 — Snapshot via np.linalg.eigh
# ══════════════════════════════════════════════════════════════════════
small_cov = D_mat.T @ D_mat                     # (N, N)  ← snapshot matrix
eigvals_eigh, eigvecs_small = np.linalg.eigh(small_cov)  # ascending order

# Map back to D-dimensional eigenfaces
eigvecs_full = D_mat @ eigvecs_small             # (D, N)

# Normalise each column to unit length
norms = np.linalg.norm(eigvecs_full, axis=0, keepdims=True)
norms[norms == 0] = 1                            # guard against zero-norm
eigvecs_full = eigvecs_full / norms

# Sort descending by eigenvalue
sort_idx = np.argsort(eigvals_eigh)[::-1]
sorted_eigvals_eigh = eigvals_eigh[sort_idx]
sorted_eigvecs_eigh = eigvecs_full[:, sort_idx]  # each column = one eigenface

# ══════════════════════════════════════════════════════════════════════
# Method 2 — SVD (verification, Tutorial 7 style)
# ══════════════════════════════════════════════════════════════════════
U, sigma, Vt = np.linalg.svd(D_mat, full_matrices=False)  # economy SVD
sorted_eigvals_svd  = sigma ** 2                # proportional to eigenvalues
sorted_eigvecs_svd  = U                         # (D, N), already descending

# ── Print eigenvalues (both methods side-by-side) ────────────────────
print('\n── Eigenvalues in descending order ─────────────────────────────────')
print(f'{"PC":>4}  {"eigh":>15}  {"SVD (σ²)":>15}')
for k in range(N):
    print(f'{k+1:>4}  {sorted_eigvals_eigh[k]:>15.4f}  {sorted_eigvals_svd[k]:>15.4f}')

# ── Cumulative explained variance plot (Tutorial 7, Cell 11) ─────────
cum_var_eigh = np.cumsum(sorted_eigvals_eigh) / np.sum(sorted_eigvals_eigh)
cum_var_svd  = np.cumsum(sorted_eigvals_svd)  / np.sum(sorted_eigvals_svd)

plt.figure()
plt.plot(np.arange(1, N+1), cum_var_eigh, '.-', label='eigh')
plt.plot(np.arange(1, N+1), cum_var_svd,  'x--', label='SVD', alpha=0.6)
plt.axhline(0.90, color='r',      linestyle='--', label='90 %')
plt.axhline(0.80, color='orange', linestyle='--', label='80 %')
plt.xlabel('Number of PCs')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance — Pug Training Data')
plt.xticks(np.arange(1, N+1))
plt.legend()
plt.grid(True)
plt.show()

# ── Normalised eigenvalue plot (Tutorial 7, Cell 12) ─────────────────
norm_eigvals = sorted_eigvals_eigh / sorted_eigvals_eigh.max()
plt.figure()
plt.plot(np.arange(1, N+1), norm_eigvals, '.-')
plt.xlabel('Eigenvalue Index')
plt.ylabel('Normalised Eigenvalue')
plt.title('Normalised Eigenvalues — Pug Training Data')
plt.xticks(np.arange(1, N+1))
plt.grid(True)
plt.show()

# ── Display eigenfaces (Tutorial 7, Cell 13) ─────────────────────────
n_show = min(N, 10)
plt.figure(figsize=(14, 4))
plt.suptitle('Eigenfaces (sorted by descending eigenvalue)', fontsize=12)
for i in range(n_show):
    plt.subplot(2, 5, i + 1)
    plt.title(f'Eigenface {i+1}', fontsize=8)
    plt.xticks([]); plt.yticks([])
    plt.imshow(sorted_eigvecs_eigh[:, i].reshape(IMG_H, IMG_W), cmap='grey')
plt.tight_layout()
plt.show()

print(f'\nTop eigenfaces stored in sorted_eigvecs_eigh : {sorted_eigvecs_eigh.shape}')
print(f'Eigenvalues stored in sorted_eigvals_eigh     : {sorted_eigvals_eigh.shape}')